In [1]:
# ============================================================
# V2 학교 교육환경 Feature Engineering 환경 설정
# - 프로젝트 루트와 external / processed 데이터 경로를 설정한다.
# - 기존 학교 위치 데이터와 향후 학교알리미 데이터를 동일한 위치에서 관리한다.
# - V1 데이터는 수정하지 않고 학교 V2 Feature를 별도 파일로 생성한다.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "프로젝트 루트에서 data 폴더를 찾을 수 없습니다."
    )


PROJECT_DIR = find_project_root()

EXTERNAL_DIR = PROJECT_DIR / "data" / "external"

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
)


print("PROJECT_DIR:", PROJECT_DIR)
print("EXTERNAL_DIR:", EXTERNAL_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR: C:\Users\uk\Apartment price
EXTERNAL_DIR: C:\Users\uk\Apartment price\data\external
PROCESSED_DIR: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed


In [2]:
# ============================================================
# 현재 보유 중인 학교 관련 외부 데이터 확인
# - 기존 학교 위치 데이터와 새로 추가된 학교알리미 파일이 있는지 확인한다.
# - 파일명을 미리 가정하지 않고 실제 external 폴더의 학교 관련 파일을 검색한다.
# - 이후 학교 위치 데이터와 교육환경 데이터를 어떤 Key로 연결할지 결정한다.
# ============================================================

school_files = [
    path
    for path in EXTERNAL_DIR.iterdir()
    if any(
        keyword in path.name.lower()
        for keyword in [
            "school",
            "학교",
            "초등",
            "중학교",
            "고등",
        ]
    )
]


print("학교 관련 파일 수:", len(school_files))

for path in sorted(school_files):
    print("-", path.name)

학교 관련 파일 수: 11
- 0001. 공통_학교속성(21).csv
- 0001. 공통_학교속성(22).csv
- 0001. 공통_학교속성(23).csv
- 0001. 공통_학교속성(24).csv
- 0001. 공통_학교속성(25).csv
- 0002. 유초중등학교개황(21).csv
- 0002. 유초중등학교개황(22).csv
- 0002. 유초중등학교개황(23).csv
- 0002. 유초중등학교개황(24).csv
- 0002. 유초중등학교개황(25).csv
- school_locations.csv


In [3]:
# ============================================================
# 기존 학교 위치 데이터 로드
# - V1에서 최근접 학교 거리 계산에 사용한 원본 학교 데이터를 확인한다.
# - 학교명, 학교급, 주소, 위도·경도 등 학교 식별에 필요한 컬럼을 점검한다.
# - 향후 학교알리미 데이터와 학교명 또는 학교코드로 연결할 수 있는지 확인한다.
# ============================================================

SCHOOL_LOCATION_PATH = (
    EXTERNAL_DIR
    / "school_locations.csv"
)


school_location = pd.read_csv(
    SCHOOL_LOCATION_PATH
)


print("shape:", school_location.shape)

print("\n컬럼:")
print(school_location.columns.tolist())

display(
    school_location.head(10)
)

shape: (12011, 18)

컬럼:
['학교ID', '학교명', '학교급구분', '설립일자', '설립형태', '본교분교구분', '운영상태', '소재지지번주소', '소재지도로명주소', '시도교육청코드', '시도교육청명', '교육지원청코드', '교육지원청명', '생성일자', '변경일자', '위도', '경도', '데이터기준일자']


,학교ID,학교명,학교급구분,설립일자,설립형태,본교분교구분,운영상태,소재지지번주소,소재지도로명주소,시도교육청코드,시도교육청명,교육지원청코드,교육지원청명,생성일자,변경일자,위도,경도,데이터기준일자
0,B000013195,경일고등학교,고등학교,1973-03-01,사립,본교,운영,경상북도 안동시 신안동 산6-2,경상북도 안동시 치이골길 46,8750000,경상북도교육청,8791000,경상북도안동교육지원청,2013-11-29,2025-10-25,36.574962,128.739295,2026-03-20
1,B000013530,보문고등학교,고등학교,1975-03-05,사립,본교,운영,광주광역시 광산구 운수동 308,광주광역시 광산구 어등대로529번길 37,7380000,광주광역시교육청,7401000,광주광역시서부교육지원청,2013-11-29,2025-10-25,35.153863,126.776870,2026-03-20
2,B000002807,대구내서초등학교,초등학교,1988-09-01,공립,본교,운영,대구광역시 서구 내당동 11-33,대구광역시 서구 달구벌대로371길 35,7240000,대구광역시교육청,7261000,대구광역시서부교육지원청,2013-11-29,2025-10-25,35.863820,128.566760,2026-03-20
3,B000007927,유림초등학교,초등학교,1932-09-26,공립,본교,운영,경상남도 함양군 유림면 화촌리 472,경상남도 함양군 유림면 천왕봉로 2927,9010000,경상남도교육청,9201000,경상남도함양교육지원청,2013-11-29,2025-10-25,35.472521,127.793074,2026-03-20
4,B000003875,호계초등학교,초등학교,1981-03-01,공립,본교,운영,경기도 안양시 동안구 호계동 923-17,경기도 안양시 동안구 경수대로707번길 27,7530000,경기도교육청,7569000,경기도안양과천교육지원청,2013-11-29,2025-10-25,37.384271,126.950123,2026-03-20
5,B000005736,조치원교동초등학교,초등학교,1945-10-05,공립,본교,운영,세종특별자치시 조치원읍 교리 1-12,세종특별자치시 조치원읍 새내18길 21,9300000,세종특별자치시교육청,9300000,세종특별자치시교육청,2013-11-29,2025-10-25,36.605228,127.297942,2026-03-20
6,B000009920,대성중학교,중학교,1951-09-01,사립,본교,운영,충청북도 청주시 흥덕구 신봉동 171-1,충청북도 청주시 흥덕구 사운로 342,8000000,충청북도교육청,8011000,충청북도청주교육지원청,2013-11-29,2025-10-25,36.658033,127.469129,2026-03-20
7,B000022777,가재울고등학교,고등학교,2013-03-01,공립,본교,운영,서울특별시 서대문구 북가좌동 476,서울특별시 서대문구 수색로 100-35,7010000,서울특별시교육청,7031000,서울특별시서부교육지원청,2013-11-29,2025-10-25,37.574446,126.909415,2026-03-20
8,B000007002,모아초등학교,초등학교,1949-12-01,공립,본교,운영,경상북도 경주시 천북면 모아리 588,경상북도 경주시 천북면 산업로 4974-5,8750000,경상북도교육청,8771000,경상북도경주교육지원청,2013-11-29,2025-10-25,35.920609,129.246622,2026-03-20
9,B000002704,반안초등학교,초등학교,2002-03-01,공립,본교,운영,부산광역시 해운대구 반여동 1164-6,부산광역시 해운대구 명장로67번길 40-42,7150000,부산광역시교육청,7211000,부산광역시해운대교육지원청,2013-11-29,2025-10-25,35.202934,129.113230,2026-03-20


## 1. 학교알리미 기본정보 및 학교코드 구축


In [4]:
# ============================================================
# 학교알리미 OpenAPI 인증키 로드
# - API 키를 노트북 코드에 직접 작성하지 않고 .env에서 불러온다.
# - GitHub 등에 인증키가 노출되는 것을 방지한다.
# - 키가 정상적으로 로드되었는지만 확인하고 실제 값은 출력하지 않는다.
# ============================================================

import os

from dotenv import load_dotenv


load_dotenv(
    PROJECT_DIR / ".env"
)

SCHOOLINFO_API_KEY = os.getenv(
    "SCHOOLINFO_API_KEY"
)


if SCHOOLINFO_API_KEY:
    print("✅ 학교알리미 API 키 로드 완료")
else:
    print("❌ SCHOOLINFO_API_KEY를 확인하세요.")

✅ 학교알리미 API 키 로드 완료


In [5]:
# ============================================================
# 학교알리미 OpenAPI 기본 설정
# - API 요청에 사용할 requests와 학교알리미 OpenAPI URL을 설정한다.
# - 실제 수집은 아래 공통 수집 함수에서 수행한다.
# - API 키는 앞 셀에서 .env를 통해 로드한 값을 사용한다.
# ============================================================

import requests


SCHOOLINFO_API_URL = (
    "https://www.schoolinfo.go.kr/openApi.do"
)


if not SCHOOLINFO_API_KEY:
    raise ValueError(
        "SCHOOLINFO_API_KEY를 .env에서 확인하세요."
    )


print("✅ 학교알리미 OpenAPI 설정 완료")


✅ 학교알리미 OpenAPI 설정 완료


In [6]:
# ============================================================
# 서울 25개 자치구 코드 정의
# - 학교알리미 신규 API 키는 시군구 코드 입력이 필수이므로
#   서울 25개 자치구를 순회하기 위한 코드 목록을 정의한다.
# - 이후 학교기본정보, 학생수, 교원수 API 수집에 공통으로 사용한다.
# ============================================================

SEOUL_GU_CODES = {
    "종로구": "11110",
    "중구": "11140",
    "용산구": "11170",
    "성동구": "11200",
    "광진구": "11215",
    "동대문구": "11230",
    "중랑구": "11260",
    "성북구": "11290",
    "강북구": "11305",
    "도봉구": "11320",
    "노원구": "11350",
    "은평구": "11380",
    "서대문구": "11410",
    "마포구": "11440",
    "양천구": "11470",
    "강서구": "11500",
    "구로구": "11530",
    "금천구": "11545",
    "영등포구": "11560",
    "동작구": "11590",
    "관악구": "11620",
    "서초구": "11650",
    "강남구": "11680",
    "송파구": "11710",
    "강동구": "11740",
}


print("자치구 수:", len(SEOUL_GU_CODES))

자치구 수: 25


In [7]:
# ============================================================
# 학교알리미 학교급 코드 정의
# - V2에서는 초등학교뿐 아니라 중학교·고등학교 교육환경도 함께 구축한다.
# - 학교급 코드를 하나의 딕셔너리로 관리해 이후 모든 API 수집에 재사용한다.
# ============================================================

SCHOOL_KIND_CODES = {
    "초등학교": "02",
    "중학교": "03",
    "고등학교": "04",
}


print("수집 학교급:")
for name, code in SCHOOL_KIND_CODES.items():
    print(f"- {name}: {code}")

수집 학교급:
- 초등학교: 02
- 중학교: 03
- 고등학교: 04


In [8]:
# ============================================================
# 기존 서울 초·중·고 학교 기준 테이블 생성
# - 기존 위치 데이터에서 서울특별시교육청 소속 운영 학교만 사용한다.
# - 초등학교·중학교·고등학교만 대상으로 한다.
# - 학교ID(B...), 학교명, 학교급, 좌표를 이후 코드 매핑 기준으로 사용한다.
# ============================================================

seoul_school_base = (
    school_location[
        (school_location["시도교육청명"] == "서울특별시교육청")
        & (school_location["학교급구분"].isin(SCHOOL_KIND_CODES))
        & (school_location["운영상태"] == "운영")
    ][
        [
            "학교ID",
            "학교명",
            "학교급구분",
            "설립형태",
            "소재지지번주소",
            "소재지도로명주소",
            "교육지원청명",
            "위도",
            "경도",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print("전체 학교 수:", len(seoul_school_base))

print("\n학교급별:")
print(
    seoul_school_base["학교급구분"]
    .value_counts()
)

print(
    "\n학교ID 중복:",
    seoul_school_base["학교ID"].duplicated().sum(),
)

전체 학교 수: 1313

학교급별:
학교급구분
초등학교    606
중학교     388
고등학교    319
Name: count, dtype: int64

학교ID 중복: 0


In [9]:
# ============================================================
# 학교급별 학교알리미 기본정보 수집 함수
# - 자치구와 학교급 코드를 받아 학교 기본정보를 조회한다.
# - 초·중·고에 동일한 수집 로직을 재사용한다.
# - 자치구와 학교급 정보를 결과에 추가해 이후 검증이 가능하도록 한다.
# ============================================================

def fetch_school_basic_info(
    gu_name,
    gu_code,
    school_kind,
    school_kind_code,
):

    params = {
        "apiKey": SCHOOLINFO_API_KEY,
        "apiType": "0",
        "sidoCode": "11",
        "sggCode": gu_code,
        "schulKndCode": school_kind_code,
    }

    response = requests.get(
        SCHOOLINFO_API_URL,
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    result = response.json()

    df = pd.DataFrame(
        result.get("list", [])
    )

    if not df.empty:
        df["자치구"] = gu_name
        df["학교급구분"] = school_kind

    return df

In [10]:
# ============================================================
# 서울 초·중·고 학교 기본정보 전체 수집
# - 서울 25개 자치구와 3개 학교급을 순회한다.
# - 총 75개 API 요청 결과를 하나의 DataFrame으로 통합한다.
# - 서버에 과도한 요청을 보내지 않도록 호출 사이에 짧은 간격을 둔다.
# ============================================================

import time


school_basic_list = []


for school_kind, school_kind_code in SCHOOL_KIND_CODES.items():

    print(f"\n===== {school_kind} =====")

    for gu_name, gu_code in SEOUL_GU_CODES.items():

        df_gu = fetch_school_basic_info(
            gu_name,
            gu_code,
            school_kind,
            school_kind_code,
        )

        print(
            f"{gu_name}: {len(df_gu)}개"
        )

        school_basic_list.append(df_gu)

        time.sleep(0.1)


school_basic_seoul = pd.concat(
    school_basic_list,
    ignore_index=True,
)


print("\n학교급별 API 조회 수:")
print(
    school_basic_seoul["학교급구분"]
    .value_counts()
)

print(
    "\nAPI 전체 학교 수:",
    len(school_basic_seoul),
)


===== 초등학교 =====


종로구: 13개

중구: 12개


용산구: 15개


성동구: 21개


...

금천구: 6개
영등포구: 9개


동작구: 8개
관악구: 17개


서초구: 12개
강남구: 21개


송파구: 21개
강동구: 14개



학교급별 API 조회 수:
학교급구분
초등학교    614
중학교     392
고등학교    323
Name: count, dtype: int64

API 전체 학교 수: 1329


In [11]:
# ============================================================
# 학교알리미 학교코드 품질 검증
# - SCHUL_CODE(S...)가 학교별로 유일한지 확인한다.
# - 학교명 + 학교급 중복도 확인해 기존 데이터와 안전하게 연결할 수 있는지 점검한다.
# ============================================================

print(
    "SCHUL_CODE 중복:",
    school_basic_seoul["SCHUL_CODE"]
    .duplicated()
    .sum(),
)


name_grade_duplicates = (
    school_basic_seoul
    .duplicated(
        subset=[
            "SCHUL_NM",
            "학교급구분",
        ]
    )
    .sum()
)


print(
    "학교명 + 학교급 중복:",
    name_grade_duplicates,
)

SCHUL_CODE 중복: 0
학교명 + 학교급 중복: 0


In [12]:
# ============================================================
# 기존 학교ID(B...) ↔ 학교알리미 학교코드(S...) 전체 매핑
# - 학교명 + 학교급을 Key로 기존 위치 데이터와 학교알리미를 연결한다.
# - B코드는 기존 공간 Feature와 연결할 때 사용하고,
#   S코드는 학교알리미 학생·학급·교원 데이터 연결에 사용한다.
# ============================================================

school_code_map = (
    seoul_school_base
    .merge(
        school_basic_seoul[
            [
                "SCHUL_CODE",
                "SCHUL_NM",
                "학교급구분",
                "자치구",
            ]
        ],
        left_on=[
            "학교명",
            "학교급구분",
        ],
        right_on=[
            "SCHUL_NM",
            "학교급구분",
        ],
        how="left",
        validate="one_to_one",
    )
)


print("기존 서울 학교:", len(seoul_school_base))

print(
    "코드 매칭 성공:",
    school_code_map["SCHUL_CODE"]
    .notna()
    .sum(),
)

print(
    "매칭 실패:",
    school_code_map["SCHUL_CODE"]
    .isna()
    .sum(),
)

기존 서울 학교: 1313
코드 매칭 성공: 1307
매칭 실패: 6


In [13]:
# ============================================================
# 학교급별 학교코드 매칭 결과 검증
# - 전체 숫자만 보지 않고 초·중·고 각각의 매칭 성공 여부를 확인한다.
# - 특정 학교급에서만 누락되는 문제가 있는지 점검한다.
# ============================================================

match_summary = (
    school_code_map
    .assign(
        매칭성공=school_code_map["SCHUL_CODE"].notna()
    )
    .groupby("학교급구분")
    .agg(
        전체학교=("학교ID", "size"),
        매칭성공=("매칭성공", "sum"),
    )
)


match_summary["매칭실패"] = (
    match_summary["전체학교"]
    - match_summary["매칭성공"]
)

match_summary["매칭률"] = (
    match_summary["매칭성공"]
    / match_summary["전체학교"]
    * 100
)


display(match_summary)

,전체학교,매칭성공,매칭실패,매칭률
학교급구분,,,,
고등학교,319,314,5,98.432602
중학교,388,387,1,99.742268
초등학교,606,606,0,100.000000


In [14]:
# ============================================================
# 학교코드 매칭 실패 학교 확인
# - 학교명 변경이나 API 표기 차이 등으로 연결되지 않은 학교를 확인한다.
# - 실패 건이 있을 경우 자동으로 넘어가지 않고 원인을 확인한 뒤 보정한다.
# ============================================================

unmatched_school = (
    school_code_map[
        school_code_map["SCHUL_CODE"].isna()
    ]
    .copy()
)


print(
    "전체 매칭 실패:",
    len(unmatched_school),
)


if len(unmatched_school) > 0:

    display(
        unmatched_school[
            [
                "학교ID",
                "학교명",
                "학교급구분",
                "소재지도로명주소",
            ]
        ]
    )

else:

    print(
        "\n✅ 서울 초·중·고 전체 학교코드 매칭 완료"
    )

전체 매칭 실패: 6


,학교ID,학교명,학교급구분,소재지도로명주소
125,B000011304,휘경공업고등학교,고등학교,서울특별시 동대문구 겸재로 21
230,B000008143,이화여자대학교사범대학부속이화·금란중학교,중학교,서울특별시 서대문구 성산로 520
382,B000011336,강서공업고등학교,고등학교,서울특별시 강서구 방화대로47길 9
777,B000011294,한양공업고등학교,고등학교,서울특별시 중구 을지로 299
791,B000011338,화곡보건경영고등학교,고등학교,서울특별시 강서구 강서로45길 70
1026,B000011340,서울항공비즈니스고등학교,고등학교,서울특별시 강서구 방화대로34길 13


In [15]:
# ============================================================
# 학교명 자동 매칭 실패 6개 학교 보정 규칙 정의
# - 중학교 1개는 특수문자 인코딩 차이로 동일 학교가 매칭되지 않았다.
# - 고등학교 5개는 교명 변경으로 기존 위치 데이터와 현재 학교알리미
#   학교명이 달라 자동 매칭되지 않았다.
# - fuzzy matching으로 임의 확정하지 않고 검증된 현재 학교명을 명시적으로 관리한다.
# ============================================================

MANUAL_SCHOOL_NAME_MAP = {
    "B000008143": "이화여자대학교사범대학부속이화?금란중학교",
    "B000011304": "서울반도체고등학교",
    "B000011336": "서울디지털콘텐츠고등학교",
    "B000011294": "한양과학기술고등학교",
    "B000011338": "서울홍신고등학교",
    "B000011340": "서울백영고등학교",
}


print("수동 보정 학교 수:", len(MANUAL_SCHOOL_NAME_MAP))

for school_id, current_name in MANUAL_SCHOOL_NAME_MAP.items():
    print(f"- {school_id}: {current_name}")

수동 보정 학교 수: 6
- B000008143: 이화여자대학교사범대학부속이화?금란중학교
- B000011304: 서울반도체고등학교
- B000011336: 서울디지털콘텐츠고등학교
- B000011294: 한양과학기술고등학교
- B000011338: 서울홍신고등학교
- B000011340: 서울백영고등학교


In [16]:
# ============================================================
# 수동 보정 학교의 현재 학교알리미 코드 확인
# - 검증한 현재 학교명을 이용해 school_basic_seoul에서 SCHUL_CODE를 조회한다.
# - S코드를 직접 하드코딩하지 않고 API 결과에서 가져와 코드 오류를 방지한다.
# ============================================================

manual_match_rows = []


for school_id, current_name in MANUAL_SCHOOL_NAME_MAP.items():

    matched = school_basic_seoul[
        school_basic_seoul["SCHUL_NM"] == current_name
    ]

    manual_match_rows.append({
        "학교ID": school_id,
        "현재학교명": current_name,
        "SCHUL_CODE": (
            matched["SCHUL_CODE"].iloc[0]
            if len(matched) == 1
            else None
        ),
        "검색건수": len(matched),
    })


manual_school_map = pd.DataFrame(
    manual_match_rows
)


display(manual_school_map)

,학교ID,현재학교명,SCHUL_CODE,검색건수
0,B000008143,이화여자대학교사범대학부속이화?금란중학교,S010001955,1
1,B000011304,서울반도체고등학교,S010000602,1
2,B000011336,서울디지털콘텐츠고등학교,S010000578,1
3,B000011294,한양과학기술고등학교,S010000601,1
4,B000011338,서울홍신고등학교,S010000655,1
5,B000011340,서울백영고등학교,S010000646,1


In [17]:
# ============================================================
# 자동 매칭 실패 학교에 수동 보정 결과 적용
# - 기존 자동 매칭 결과는 그대로 유지한다.
# - SCHUL_CODE가 없는 6개 학교에만 검증된 현재 학교코드를 채운다.
# - 기존 학교ID(B...)와 현재 학교알리미 코드(S...) 관계를 보존한다.
# - 자동 매칭 단계에서 붙지 않은 자치구도 학교알리미 기본정보로 채운다.
# ============================================================

manual_code_dict = dict(
    zip(
        manual_school_map["학교ID"],
        manual_school_map["SCHUL_CODE"],
    )
)


manual_name_dict = dict(
    zip(
        manual_school_map["학교ID"],
        manual_school_map["현재학교명"],
    )
)


missing_mask = school_code_map["SCHUL_CODE"].isna()


school_code_map.loc[
    missing_mask,
    "SCHUL_CODE",
] = (
    school_code_map.loc[
        missing_mask,
        "학교ID",
    ]
    .map(manual_code_dict)
)


school_code_map.loc[
    missing_mask,
    "SCHUL_NM",
] = (
    school_code_map.loc[
        missing_mask,
        "학교ID",
    ]
    .map(manual_name_dict)
)


manual_gu_dict = dict(
    zip(
        school_basic_seoul["SCHUL_CODE"],
        school_basic_seoul["자치구"],
    )
)


school_code_map.loc[
    missing_mask,
    "자치구",
] = (
    school_code_map.loc[
        missing_mask,
        "SCHUL_CODE",
    ]
    .map(manual_gu_dict)
)

In [18]:
# ============================================================
# 서울 초·중·고 학교코드 최종 검증
# - 자동 매칭과 수동 보정을 모두 적용한 결과를 학교급별로 확인한다.
# - 최종 목표는 기존 1,313개 학교 모두 SCHUL_CODE를 갖는 것이다.
# - 이후 학생수·학급수·교원수 수집은 이 매핑을 기준으로 진행한다.
# ============================================================

match_summary_final = (
    school_code_map
    .assign(
        매칭성공=school_code_map["SCHUL_CODE"].notna()
    )
    .groupby("학교급구분")
    .agg(
        전체학교=("학교ID", "size"),
        매칭성공=("매칭성공", "sum"),
    )
)


match_summary_final["매칭실패"] = (
    match_summary_final["전체학교"]
    - match_summary_final["매칭성공"]
)

match_summary_final["매칭률"] = (
    match_summary_final["매칭성공"]
    / match_summary_final["전체학교"]
    * 100
)


display(match_summary_final)


print(
    "\n전체 학교:",
    len(school_code_map),
)

print(
    "전체 매칭 실패:",
    school_code_map["SCHUL_CODE"].isna().sum(),
)


print(
    "자치구 결측:",
    school_code_map["자치구"].isna().sum(),
)


assert school_code_map["SCHUL_CODE"].isna().sum() == 0
assert school_code_map["자치구"].isna().sum() == 0

print(
    "\n✅ 서울 초·중·고 전체 학교코드 매칭 완료"
)

,전체학교,매칭성공,매칭실패,매칭률
학교급구분,,,,
고등학교,319,319,0,100.0
중학교,388,388,0,100.0
초등학교,606,606,0,100.0



전체 학교: 1313
전체 매칭 실패: 0
자치구 결측: 0

✅ 서울 초·중·고 전체 학교코드 매칭 완료


In [19]:
# ============================================================
# 서울 초·중·고 학교코드 매핑 저장
# - 기존 위치 데이터의 학교ID(B...)와 학교알리미 SCHUL_CODE(S...) 매핑을 저장한다.
# - 이후 학생수·학급수·교원수 데이터를 SCHUL_CODE로 연결할 때 재사용한다.
# - 학교명 변경 보정까지 완료된 1,313개 학교가 모두 포함되어야 한다.
# ============================================================

SCHOOL_CODE_MAP_PATH = (
    PROCESSED_DIR
    / "school_code_map_v2.parquet"
)


school_code_map.to_parquet(
    SCHOOL_CODE_MAP_PATH,
    index=False,
)


print("저장 완료:", SCHOOL_CODE_MAP_PATH)
print("shape:", school_code_map.shape)

print(
    "SCHUL_CODE 결측:",
    school_code_map["SCHUL_CODE"].isna().sum(),
)

print(
    "SCHUL_CODE 중복:",
    school_code_map["SCHUL_CODE"].duplicated().sum(),
)

저장 완료: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed\school_code_map_v2.parquet
shape: (1313, 12)
SCHUL_CODE 결측: 0
SCHUL_CODE 중복: 0


## 2. SchoolInfo 2024~2026 학생·학급 데이터 구축


In [20]:
# ============================================================
# 학교알리미 학생·학급 데이터 수집 함수
# - 공시연도, 자치구, 학교급을 받아 apiType 09 데이터를 조회한다.
# - 이후 2024~2026년 서울 초·중·고 전체에 동일한 함수를 재사용한다.
# - 결과에 기준연도·자치구·학교급을 명시적으로 추가한다.
# ============================================================

def fetch_school_environment(
    year,
    gu_name,
    gu_code,
    school_kind,
    school_kind_code,
):

    params = {
        "apiKey": SCHOOLINFO_API_KEY,
        "apiType": "09",
        "sidoCode": "11",
        "sggCode": gu_code,
        "schulKndCode": school_kind_code,
        "pbanYr": str(year),
    }

    response = requests.get(
        SCHOOLINFO_API_URL,
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    result = response.json()

    if result.get("resultCode") == "fail":
        print(
            f"⚠️ {year} {gu_name} {school_kind}:",
            result.get("resultMsg"),
        )
        return pd.DataFrame()

    df = pd.DataFrame(
        result.get("list", [])
    )

    if not df.empty:
        df["기준연도"] = year
        df["자치구"] = gu_name
        df["학교급구분"] = school_kind

    return df

In [21]:
# ============================================================
# 2024~2026 서울 초·중·고 교육환경 데이터 전체 수집
# - 3개 연도 × 3개 학교급 × 서울 25개 자치구를 순회한다.
# - 각 요청 결과를 하나의 연도별 학교 교육환경 데이터로 통합한다.
# - API 서버 부담을 줄이기 위해 요청 사이에 짧은 간격을 둔다.
# ============================================================

SCHOOL_YEARS = [
    2024,
    2025,
    2026,
]


school_environment_list = []


for year in SCHOOL_YEARS:

    print(f"\n========== {year} ==========")

    for school_kind, school_kind_code in SCHOOL_KIND_CODES.items():

        total_count = 0

        for gu_name, gu_code in SEOUL_GU_CODES.items():

            df_gu = fetch_school_environment(
                year,
                gu_name,
                gu_code,
                school_kind,
                school_kind_code,
            )

            total_count += len(df_gu)

            if not df_gu.empty:
                school_environment_list.append(
                    df_gu
                )

            time.sleep(0.1)

        print(
            f"{school_kind}: {total_count}개"
        )


school_environment_raw = pd.concat(
    school_environment_list,
    ignore_index=True,
)


print(
    "\n전체 수집 행:",
    len(school_environment_raw),
)


========== 2024 ==========


초등학교: 605개


중학교: 388개


고등학교: 318개

========== 2025 ==========


초등학교: 606개


중학교: 388개


고등학교: 318개

========== 2026 ==========


초등학교: 606개


중학교: 389개


고등학교: 319개

전체 수집 행: 3937


In [22]:
# ============================================================
# 학교 교육환경 데이터 수집 결과 검증
# - 연도별·학교급별 학교 수를 확인한다.
# - 특정 연도 또는 학교급의 데이터가 비정상적으로 누락되지 않았는지 점검한다.
# - 학교 신설·폐교·교명 변경 때문에 연도별 학교 수는 조금 다를 수 있다.
# ============================================================

school_count_summary = (
    school_environment_raw
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    .agg(
        학교수=("SCHUL_CODE", "nunique"),
        행수=("SCHUL_CODE", "size"),
    )
)


display(
    school_count_summary
)

학교수   행수
기준연도 학교급구분          
2024 고등학교   318  318
     중학교    388  388
     초등학교   605  605
2025 고등학교   318  318
     중학교    388  388
     초등학교   606  606
2026 고등학교   319  319
     중학교    389  389
     초등학교   606  606

In [23]:
# ============================================================
# 2026 학교알리미와 기존 학교코드 매핑 차이 확인
# - 현재 위치 기준 학교는 1,313개지만 2026 API는 1,314개다.
# - 어느 학교가 API에만 존재하거나 기존 위치 데이터에만 존재하는지 확인한다.
# - 학교 환경 Feature와 공간 위치를 연결하기 전에 반드시 원인을 확인한다.
# ============================================================

api_2026_codes = set(
    school_environment_raw.loc[
        school_environment_raw["기준연도"] == 2026,
        "SCHUL_CODE",
    ]
)

mapped_codes = set(
    school_code_map["SCHUL_CODE"]
)


api_only_codes = (
    api_2026_codes
    - mapped_codes
)

map_only_codes = (
    mapped_codes
    - api_2026_codes
)


print(
    "2026 API에만 존재:",
    len(api_only_codes),
)

print(
    "기존 매핑에만 존재:",
    len(map_only_codes),
)


if api_only_codes:

    display(
        school_environment_raw[
            (school_environment_raw["기준연도"] == 2026)
            & (
                school_environment_raw["SCHUL_CODE"]
                .isin(api_only_codes)
            )
        ][
            [
                "SCHUL_CODE",
                "SCHUL_NM",
                "학교급구분",
                "자치구",
            ]
        ]
    )


if map_only_codes:

    display(
        school_code_map[
            school_code_map["SCHUL_CODE"]
            .isin(map_only_codes)
        ][
            [
                "학교ID",
                "학교명",
                "학교급구분",
                "SCHUL_CODE",
            ]
        ]
    )

2026 API에만 존재: 2
기존 매핑에만 존재: 1


,SCHUL_CODE,SCHUL_NM,학교급구분,자치구
3422,S010006282,아현중학교부설방송통신중학교,중학교,마포구
3433,S010000993,신화중학교,중학교,양천구


,학교ID,학교명,학교급구분,SCHUL_CODE
1053,B000008124,연서중학교,중학교,S010001947


In [24]:
# ============================================================
# 연도별 학교코드 매칭률 확인
# - 2024~2026 교육환경 데이터가 현재 학교 위치 데이터와
#   SCHUL_CODE 기준으로 얼마나 연결되는지 확인한다.
# - 폐교·신설학교 때문에 과거 연도는 100%가 아닐 수 있다.
# ============================================================

school_environment_coverage = (
    school_environment_raw
    .assign(
        위치매핑여부=lambda x:
            x["SCHUL_CODE"].isin(mapped_codes)
    )
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    .agg(
        전체학교=("SCHUL_CODE", "size"),
        위치매핑=("위치매핑여부", "sum"),
    )
)


school_environment_coverage["매칭률"] = (
    school_environment_coverage["위치매핑"]
    / school_environment_coverage["전체학교"]
    * 100
)


display(
    school_environment_coverage
)

전체학교  위치매핑         매칭률
기준연도 학교급구분                        
2024 고등학교    318   318  100.000000
     중학교     388   387   99.742268
     초등학교    605   604   99.834711
2025 고등학교    318   318  100.000000
     중학교     388   387   99.742268
     초등학교    606   606  100.000000
2026 고등학교    319   319  100.000000
     중학교     389   387   99.485861
     초등학교    606   606  100.000000

In [25]:
# ============================================================
# 일반 주거 학군 분석 대상 학교 필터링
# - 학교알리미 API에는 일반 초·중·고 외에 부설 방송통신학교가 포함될 수 있다.
# - 주거지 교육환경을 측정하는 목적과 맞지 않는 방송통신학교는 제외한다.
# - 일반 초등학교·중학교·고등학교의 교육환경만 유지한다.
# ============================================================

EXCLUDE_SCHOOL_CODES = {
    "S010006282",  # 아현중학교부설방송통신중학교
}


school_environment_filtered = (
    school_environment_raw[
        ~school_environment_raw["SCHUL_CODE"]
        .isin(EXCLUDE_SCHOOL_CODES)
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "제외 전:",
    len(school_environment_raw),
)

print(
    "제외 후:",
    len(school_environment_filtered),
)

print("\n연도별·학교급별 학교 수:")

display(
    school_environment_filtered
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    ["SCHUL_CODE"]
    .nunique()
    .to_frame("학교수")
)

제외 전: 3937
제외 후: 3934

연도별·학교급별 학교 수:


학교수
기준연도 학교급구분     
2024 고등학교   318
     중학교    387
     초등학교   605
2025 고등학교   318
     중학교    387
     초등학교   606
2026 고등학교   319
     중학교    388
     초등학교   606

In [26]:
# ============================================================
# 위치정보가 없는 정상 운영 학교 확인
# - 현재 위치 데이터에 없는 학교 중 실제 분석 대상 학교를 찾는다.
# - 신설·재개교 학교는 제외하지 않고 학교알리미 기본정보에서
#   좌표를 보완해 공간 Feature 계산에 사용할 예정이다.
# ============================================================

mapped_codes = set(
    school_code_map["SCHUL_CODE"]
)

environment_codes = set(
    school_environment_filtered["SCHUL_CODE"]
)


unmapped_environment_codes = (
    environment_codes
    - mapped_codes
)


print(
    "위치 매핑이 필요한 학교:",
    len(unmapped_environment_codes),
)


display(
    school_environment_filtered[
        school_environment_filtered["SCHUL_CODE"]
        .isin(unmapped_environment_codes)
    ][
        [
            "기준연도",
            "SCHUL_CODE",
            "SCHUL_NM",
            "학교급구분",
            "자치구",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "기준연도",
            "학교급구분",
        ]
    )
)

위치 매핑이 필요한 학교: 2


,기준연도,SCHUL_CODE,SCHUL_NM,학교급구분,자치구
27,2024,S010002379,서울보광초등학교,초등학교,용산구
3430,2026,S010000993,신화중학교,중학교,양천구


In [27]:
# ============================================================
# 누락 학교 2개의 위치정보 보완
# - 학교알리미 기본정보의 위·경도를 사용해 기존 위치 데이터 누락을 보완한다.
# - 별도 지오코딩 없이 공식 API 좌표를 그대로 활용한다.
# - SCHUL_CODE를 기준으로 이후 연도별 학교환경 데이터와 연결한다.
# ============================================================

school_location_patch = (
    school_basic_seoul[
        school_basic_seoul["SCHUL_CODE"].isin(
            [
                "S010002379",
                "S010000993",
            ]
        )
    ][
        [
            "SCHUL_CODE",
            "SCHUL_NM",
            "학교급구분",
            "자치구",
            "LTTUD",
            "LGTUD",
        ]
    ]
    .rename(
        columns={
            "SCHUL_NM": "학교명",
            "LTTUD": "위도",
            "LGTUD": "경도",
        }
    )
    .copy()
)


school_location_patch[
    ["위도", "경도"]
] = (
    school_location_patch[
        ["위도", "경도"]
    ]
    .apply(pd.to_numeric, errors="coerce")
)


display(school_location_patch)

print(
    "\n좌표 결측:",
    school_location_patch[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
)

,SCHUL_CODE,학교명,학교급구분,자치구,위도,경도
27,S010002379,서울보광초등학교,초등학교,용산구,37.532651,126.996461
819,S010000993,신화중학교,중학교,양천구,37.541096,126.830105



좌표 결측: 위도    0
경도    0
dtype: int64


In [28]:
# ============================================================
# V2 학교 위치 기준 테이블 완성
# - 기존 school_code_map의 1,313개 학교 위치정보를 기본으로 사용한다.
# - 기존 위치 원본에 없던 서울보광초와 신화중 2개를 학교알리미 좌표로 추가한다.
# - SCHUL_CODE를 학교환경 데이터와 공간정보를 연결하는 기준 Key로 사용한다.
# ============================================================

school_location_v2 = (
    school_code_map[
        [
            "학교ID",
            "SCHUL_CODE",
            "SCHUL_NM",
            "학교급구분",
            "자치구",
            "위도",
            "경도",
        ]
    ]
    .rename(
        columns={
            "SCHUL_NM": "학교명",
        }
    )
    .copy()
)


school_location_patch["학교ID"] = pd.NA

school_location_patch = school_location_patch[
    [
        "학교ID",
        "SCHUL_CODE",
        "학교명",
        "학교급구분",
        "자치구",
        "위도",
        "경도",
    ]
]


school_location_v2 = pd.concat(
    [
        school_location_v2,
        school_location_patch,
    ],
    ignore_index=True,
)


print("학교 위치 전체:", len(school_location_v2))

print(
    "SCHUL_CODE 중복:",
    school_location_v2["SCHUL_CODE"].duplicated().sum(),
)

print(
    "좌표 결측:",
    school_location_v2[
        ["위도", "경도"]
    ].isna().sum().sum(),
)

학교 위치 전체: 1315
SCHUL_CODE 중복: 0
좌표 결측: 0


In [29]:
# ============================================================
# 연도별 학교환경 위치 매핑 최종 검증
# - 방송통신중학교를 제외한 2024~2026 일반 초·중·고 학교가
#   모두 공간 좌표와 연결되는지 확인한다.
# - 최종적으로 모든 연도·학교급에서 위치 매핑률 100%를 목표로 한다.
# ============================================================

location_codes = set(
    school_location_v2["SCHUL_CODE"]
)


school_environment_coverage_final = (
    school_environment_filtered
    .assign(
        위치매핑여부=lambda x:
            x["SCHUL_CODE"].isin(location_codes)
    )
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    .agg(
        전체학교=("SCHUL_CODE", "size"),
        위치매핑=("위치매핑여부", "sum"),
    )
)


school_environment_coverage_final["매칭률"] = (
    school_environment_coverage_final["위치매핑"]
    / school_environment_coverage_final["전체학교"]
    * 100
)


display(
    school_environment_coverage_final
)


assert (
    school_environment_coverage_final["매칭률"]
    == 100
).all()


print(
    "\n✅ 2024~2026 학교환경 위치 매핑 완료"
)

전체학교  위치매핑    매칭률
기준연도 학교급구분                   
2024 고등학교    318   318  100.0
     중학교     387   387  100.0
     초등학교    605   605  100.0
2025 고등학교    318   318  100.0
     중학교     387   387  100.0
     초등학교    606   606  100.0
2026 고등학교    319   319  100.0
     중학교     388   388  100.0
     초등학교    606   606  100.0


✅ 2024~2026 학교환경 위치 매핑 완료


In [30]:
# ============================================================
# SchoolInfo 2024~2026 학교환경 기준 테이블 정제
# - 교원 정의가 불명확한 TEACH_CNT / TEACH_CAL은 최종 Feature에서 제외한다.
# - SchoolInfo와 EDSS에서 의미가 동일하게 확인된 학생수·학급수만 우선 정제한다.
# - 학교코드와 좌표를 연결해 2024~2026 Crosswalk 검증용 기준 테이블을 만든다.
# ============================================================

school_environment_clean = (
    school_environment_filtered[
        [
            "기준연도",
            "SCHUL_CODE",
            "SCHUL_NM",
            "학교급구분",
            "자치구",
            "FOND_SC_CODE",
            "COL_S_SUM",
            "COL_C_SUM",
        ]
    ]
    .rename(
        columns={
            "SCHUL_NM": "학교명",
            "FOND_SC_CODE": "설립형태",
            "COL_S_SUM": "학생수",
            "COL_C_SUM": "학급수",
        }
    )
    .copy()
)


school_environment_clean[
    ["학생수", "학급수"]
] = (
    school_environment_clean[
        ["학생수", "학급수"]
    ]
    .apply(pd.to_numeric, errors="coerce")
)


school_environment_clean["학급당학생수"] = (
    school_environment_clean["학생수"]
    / school_environment_clean["학급수"]
)


school_environment_clean = (
    school_environment_clean
    .merge(
        school_location_v2[
            [
                "SCHUL_CODE",
                "위도",
                "경도",
            ]
        ],
        on="SCHUL_CODE",
        how="left",
        validate="many_to_one",
    )
)


print(
    "shape:",
    school_environment_clean.shape,
)

print(
    "Key 중복:",
    school_environment_clean
    .duplicated(
        subset=[
            "기준연도",
            "SCHUL_CODE",
        ]
    )
    .sum(),
)

print(
    "좌표 결측:",
    school_environment_clean[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
    .sum(),
)


assert (
    school_environment_clean
    .duplicated(
        subset=[
            "기준연도",
            "SCHUL_CODE",
        ]
    )
    .sum()
    == 0
)

assert (
    school_environment_clean[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


SCHOOL_ENV_2024_2026_PATH = (
    PROCESSED_DIR
    / "school_environment_2024_2026.parquet"
)


school_environment_clean.to_parquet(
    SCHOOL_ENV_2024_2026_PATH,
    index=False,
)


print(
    "✅ SchoolInfo 2024~2026 기준 테이블 저장 완료:",
    SCHOOL_ENV_2024_2026_PATH,
)


shape: (3934, 11)
Key 중복: 0
좌표 결측: 0


✅ SchoolInfo 2024~2026 기준 테이블 저장 완료: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed\school_environment_2024_2026.parquet


## 3. EDSS 2021~2025 원본 7종 통합


In [31]:
# ============================================================
# EDSS 2021~2025 원본 파일 자동 탐색
# - 학교환경 관련 7종 데이터의 연도별 원본 파일을 확인한다.
# - 각 데이터가 2021~2025 총 5개씩 존재하는지 검증한다.
# - 원본 CSV는 external에 그대로 보존한다.
# ============================================================

EDSS_SEARCH_ROOT = EXTERNAL_DIR

EDSS_FILE_GROUPS = {
    "학교속성": "0001. 공통_학교속성",
    "학교개황": "0002. 유초중등학교개황",
    "학급현황": "0003. 유초중등학급현황",
    "교원현황": "0007. 유초중등교원현황",
    "도서관운영현황": "0008. 도서관운영현황",
    "건물현황": "0105. 건물현황",
    "학생변동현황": "0210. 학생변동현황",
}


for group_name, file_prefix in EDSS_FILE_GROUPS.items():

    files = sorted(
        EDSS_SEARCH_ROOT.rglob(
            f"{file_prefix}*.csv"
        )
    )

    print(
        f"{group_name}: {len(files)}개"
    )

학교속성: 5개
학교개황: 5개
학급현황: 5개
교원현황: 5개
도서관운영현황: 5개
건물현황: 5개
학생변동현황: 5개


In [32]:
# ============================================================
# EDSS 연도별 CSV 통합 함수
# - 동일 종류의 2021~2025 CSV를 하나의 DataFrame으로 합친다.
# - 연도별 컬럼 차이가 있으면 전체 컬럼을 유지하면서 통합한다.
# - 조사년도 + 개방ID가 있으면 시계열 순서로 정렬한다.
# ============================================================

import re


def read_edss_csv(file_path):

    for encoding in ["cp949", "utf-8-sig", "utf-8"]:

        try:
            return pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False,
            )

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"인코딩 확인 필요: {file_path.name}"
    )


def merge_edss_files(file_prefix):

    pattern = re.compile(
        rf"{re.escape(file_prefix)}\((21|22|23|24|25)\)\.csv$"
    )

    files = sorted(
        path
        for path in EDSS_SEARCH_ROOT.rglob("*.csv")
        if pattern.fullmatch(path.name)
    )

    if len(files) != 5:
        raise ValueError(
            f"{file_prefix}: 파일 수 {len(files)}개"
        )

    dataframes = [
        read_edss_csv(file_path)
        for file_path in files
    ]

    merged = pd.concat(
        dataframes,
        ignore_index=True,
        sort=False,
    )

    sort_cols = [
        col
        for col in ["조사년도", "개방ID"]
        if col in merged.columns
    ]

    if sort_cols:
        merged = (
            merged
            .sort_values(sort_cols)
            .reset_index(drop=True)
        )

    return merged

In [33]:
# ============================================================
# EDSS 학교환경 7종 통합 및 Parquet 저장
# - 연도별 35개 CSV를 데이터 종류별 7개 Parquet으로 통합한다.
# - external의 원본 CSV는 수정하지 않는다.
# - 이후 분석에서는 processed/edss의 Parquet만 사용한다.
# ============================================================

EDSS_PROCESSED_DIR = (
    PROCESSED_DIR
    / "edss"
)

EDSS_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


EDSS_OUTPUT_NAMES = {
    "학교속성": "school_attribute_2021_2025.parquet",
    "학교개황": "school_overview_2021_2025.parquet",
    "학급현황": "class_status_2021_2025.parquet",
    "교원현황": "teacher_status_2021_2025.parquet",
    "도서관운영현황": "library_status_2021_2025.parquet",
    "건물현황": "building_status_2021_2025.parquet",
    "학생변동현황": "student_movement_2021_2025.parquet",
}


for group_name, file_prefix in EDSS_FILE_GROUPS.items():

    merged = merge_edss_files(
        file_prefix
    )

    output_path = (
        EDSS_PROCESSED_DIR
        / EDSS_OUTPUT_NAMES[group_name]
    )

    merged.to_parquet(
        output_path,
        index=False,
    )

    print(
        f"{group_name}: "
        f"{merged.shape} → {output_path.name}"
    )


print("\n✅ EDSS 7종 저장 완료")

학교속성: (104528, 30) → school_attribute_2021_2025.parquet


학교개황: (104528, 79) → school_overview_2021_2025.parquet


학급현황: (104453, 49) → class_status_2021_2025.parquet


교원현황: (104528, 51) → teacher_status_2021_2025.parquet


도서관운영현황: (61171, 15) → library_status_2021_2025.parquet


건물현황: (334041, 28) → building_status_2021_2025.parquet


학생변동현황: (550512, 50) → student_movement_2021_2025.parquet

✅ EDSS 7종 저장 완료


In [34]:
# ============================================================
# EDSS 학교환경 7종 통합 Parquet 로드
# - 이후 Feature Engineering에서는 연도별 CSV를 다시 사용하지 않는다.
# - processed에 저장한 2021~2025 통합 데이터만 사용한다.
# ============================================================

edss_school_attribute = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "school_attribute_2021_2025.parquet"
)

edss_school_overview = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "school_overview_2021_2025.parquet"
)

edss_class_status = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "class_status_2021_2025.parquet"
)

edss_teacher_status = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "teacher_status_2021_2025.parquet"
)

edss_library_status = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "library_status_2021_2025.parquet"
)

edss_building_status = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "building_status_2021_2025.parquet"
)

edss_student_movement = pd.read_parquet(
    EDSS_PROCESSED_DIR
    / "student_movement_2021_2025.parquet"
)


for name, df in {
    "학교속성": edss_school_attribute,
    "학교개황": edss_school_overview,
    "학급현황": edss_class_status,
    "교원현황": edss_teacher_status,
    "도서관": edss_library_status,
    "건물": edss_building_status,
    "학생변동": edss_student_movement,
}.items():

    print(
        f"{name}: {df.shape}"
    )

학교속성: (104528, 30)
학교개황: (104528, 79)
학급현황: (104453, 49)
교원현황: (104528, 51)
도서관: (61171, 15)
건물: (334041, 28)
학생변동: (550512, 50)


## 4. EDSS 학교환경 기준 테이블 및 Crosswalk


In [35]:
# ============================================================
# 서울 초·중·고 EDSS 학교 기준정보 추출
# - 학교속성에서 서울 소재 학교만 남긴다.
# - 학제유형명을 초등학교·중학교·고등학교 3개 수준으로 통합한다.
# - 고등학교의 세부 유형은 하나의 고등학교 학교급으로 묶는다.
# ============================================================

HIGH_SCHOOL_TYPES = {
    "일반고등학교",
    "특성화고등학교",
    "특수목적고등학교",
    "자율고등학교",
}


def classify_school_level(school_type):

    if school_type == "초등학교":
        return "초등학교"

    if school_type == "중학교":
        return "중학교"

    if school_type in HIGH_SCHOOL_TYPES:
        return "고등학교"

    return pd.NA


edss_school_base = (
    edss_school_attribute[
        edss_school_attribute["시도명"] == "서울"
    ]
    .copy()
)


edss_school_base["학교급구분"] = (
    edss_school_base["학제유형명"]
    .apply(classify_school_level)
)


edss_school_base = (
    edss_school_base[
        edss_school_base["학교급구분"].notna()
    ][
        [
            "조사년도",
            "개방ID",
            "시군명",
            "학교급구분",
            "설립구분명",
        ]
    ]
    .rename(
        columns={
            "조사년도": "기준연도",
            "시군명": "자치구",
            "설립구분명": "설립형태",
        }
    )
    .reset_index(drop=True)
)


display(
    edss_school_base.head()
)

,기준연도,개방ID,자치구,학교급구분,설립형태
0,2021,1000279486,관악구,초등학교,공립
1,2021,1018460035,송파구,초등학교,공립
2,2021,1018780291,서대문구,고등학교,공립
3,2021,1019676685,영등포구,초등학교,공립
4,2021,1021709263,관악구,중학교,사립


In [36]:
# ============================================================
# EDSS 학교개황에서 학생수·교원수 추출
# - 학교개황의 총 학생수와 총 교원수만 우선 가져온다.
# - 시설 관련 컬럼은 이후 EDA에서 필요성을 판단한 뒤 추가한다.
# ============================================================

edss_overview_core = (
    edss_school_overview[
        [
            "조사년도",
            "개방ID",
            "유초중등학교개황_학생수",
            "유초중등학교개황_교원수",
        ]
    ]
    .rename(
        columns={
            "조사년도": "기준연도",
            "유초중등학교개황_학생수": "학생수",
            "유초중등학교개황_교원수": "교원수",
        }
    )
    .copy()
)

In [37]:
# ============================================================
# EDSS 학급현황에서 총 학급수 추출
# - 학급수는 별도 학급현황 데이터의 유초중등학급_학급수를 사용한다.
# - 학교개황 내부의 학급수 컬럼과 혼용하지 않는다.
# ============================================================

edss_class_core = (
    edss_class_status[
        [
            "조사년도",
            "개방ID",
            "유초중등학급_학급수",
        ]
    ]
    .rename(
        columns={
            "조사년도": "기준연도",
            "유초중등학급_학급수": "학급수",
        }
    )
    .copy()
)

In [38]:
# ============================================================
# EDSS 학교환경 2021~2025 통합
# - 학교속성·학교개황·학급현황을 기준연도 + 개방ID로 결합한다.
# - 학생수·학급수·교원수와 자치구·학교급을 하나의 학교 단위 테이블로 만든다.
# - 각 원본에서 Key가 유일하다는 전제로 one-to-one 검증을 수행한다.
# ============================================================

edss_school_environment = (
    edss_school_base
    .merge(
        edss_overview_core,
        on=[
            "기준연도",
            "개방ID",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        edss_class_core,
        on=[
            "기준연도",
            "개방ID",
        ],
        how="left",
        validate="one_to_one",
    )
)


numeric_cols = [
    "학생수",
    "학급수",
    "교원수",
]

edss_school_environment[numeric_cols] = (
    edss_school_environment[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
)


print(
    "shape:",
    edss_school_environment.shape,
)

print("\n결측치:")
print(
    edss_school_environment[
        numeric_cols
    ]
    .isna()
    .sum()
)

shape: (6594, 8)

결측치:
학생수    0
학급수    0
교원수    0
dtype: int64


In [39]:
# ============================================================
# EDSS 2021~2025 활성 학교 현황 확인
# - 결측치는 없지만 휴교 등으로 학생수·학급수·교원수가 0인 학교가 있을 수 있다.
# - 실제 교육환경 Feature에 사용할 학교는 세 값이 모두 양수인 학교로 정의한다.
# - 연도·학교급별 학교 수를 확인해 SchoolInfo와 비교할 준비를 한다.
# ============================================================

edss_school_environment_active = (
    edss_school_environment[
        (edss_school_environment["학생수"] > 0)
        & (edss_school_environment["학급수"] > 0)
        & (edss_school_environment["교원수"] > 0)
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "전체:",
    len(edss_school_environment),
)

print(
    "활성 학교:",
    len(edss_school_environment_active),
)

print(
    "제외:",
    len(edss_school_environment)
    - len(edss_school_environment_active),
)


display(
    edss_school_environment_active
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    ["개방ID"]
    .nunique()
    .to_frame("학교수")
)

전체: 6594
활성 학교: 6561
제외: 33


학교수
기준연도 학교급구분     
2021 고등학교   320
     중학교    388
     초등학교   603
2022 고등학교   321
     중학교    389
     초등학교   605
2023 고등학교   321
     중학교    389
     초등학교   604
2024 고등학교   318
     중학교    387
     초등학교   605
2025 고등학교   318
     중학교    387
     초등학교   606

In [40]:
# ============================================================
# EDSS와 SchoolInfo 겹치는 연도의 학교 수 비교
# - 2024·2025는 두 데이터 소스가 모두 존재하므로 교차검증에 사용한다.
# - 연도·학교급별 학교 수가 비슷한지 먼저 확인한 뒤 학교 단위 매칭을 진행한다.
# ============================================================

edss_overlap_count = (
    edss_school_environment_active[
        edss_school_environment_active["기준연도"]
        .isin([2024, 2025])
    ]
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    ["개방ID"]
    .nunique()
    .rename("EDSS")
)


schoolinfo_overlap_count = (
    school_environment_clean[
        school_environment_clean["기준연도"]
        .isin([2024, 2025])
    ]
    .groupby(
        [
            "기준연도",
            "학교급구분",
        ]
    )
    ["SCHUL_CODE"]
    .nunique()
    .rename("SchoolInfo")
)


overlap_count = pd.concat(
    [
        edss_overlap_count,
        schoolinfo_overlap_count,
    ],
    axis=1,
)


overlap_count["차이"] = (
    overlap_count["EDSS"]
    - overlap_count["SchoolInfo"]
)


display(overlap_count)

EDSS  SchoolInfo  차이
기준연도 학교급구분                      
2024 고등학교    318         318   0
     중학교     387         387   0
     초등학교    605         605   0
2025 고등학교    318         318   0
     중학교     387         387   0
     초등학교    606         606   0

In [41]:
# ============================================================
# EDSS ↔ SchoolInfo 매칭 Key 정의
# - EDSS와 SchoolInfo에서 정의가 동일하게 확인된 학생수·학급수만 사용한다.
# - 연도·자치구·학교급을 함께 사용해 학교 식별 정확도를 높인다.
# ============================================================

MATCH_COLS = [
    "기준연도",
    "자치구",
    "학교급구분",
    "학생수",
    "학급수",
]


edss_overlap = (
    edss_school_environment_active[
        edss_school_environment_active["기준연도"]
        .isin([2024, 2025])
    ]
    .copy()
)


schoolinfo_overlap = (
    school_environment_clean[
        school_environment_clean["기준연도"]
        .isin([2024, 2025])
    ]
    .copy()
)


print(
    "EDSS 중복 조합:",
    edss_overlap
    .duplicated(
        subset=MATCH_COLS,
        keep=False,
    )
    .sum(),
)

print(
    "SchoolInfo 중복 조합:",
    schoolinfo_overlap
    .duplicated(
        subset=MATCH_COLS,
        keep=False,
    )
    .sum(),
)

EDSS 중복 조합: 8
SchoolInfo 중복 조합: 8


In [42]:
# ============================================================
# EDSS ↔ SchoolInfo 학생수·학급수 기반 exact match
# - 동일 연도·자치구·학교급·학생수·학급수 조합이 유일한 SchoolInfo 학교만 자동 연결한다.
# - 중복 조합은 자동으로 확정하지 않고 이후 수동 Crosswalk 대상으로 남긴다.
# - 결과를 crosswalk_v2로 저장해 연도 간 일관성 검증에 사용한다.
# ============================================================

schoolinfo_match_counts = (
    schoolinfo_overlap
    .groupby(MATCH_COLS)
    .size()
    .rename("후보수")
    .reset_index()
)


schoolinfo_unique = (
    schoolinfo_overlap
    .merge(
        schoolinfo_match_counts,
        on=MATCH_COLS,
        how="left",
    )
)


schoolinfo_unique = (
    schoolinfo_unique[
        schoolinfo_unique["후보수"] == 1
    ]
    .copy()
)


crosswalk_v2 = (
    edss_overlap
    .merge(
        schoolinfo_unique[
            [
                *MATCH_COLS,
                "SCHUL_CODE",
                "학교명",
            ]
        ],
        on=MATCH_COLS,
        how="left",
    )
)


print(
    "EDSS 대상:",
    len(crosswalk_v2),
)

print(
    "exact match:",
    crosswalk_v2["SCHUL_CODE"]
    .notna()
    .sum(),
)

print(
    "매칭 실패:",
    crosswalk_v2["SCHUL_CODE"]
    .isna()
    .sum(),
)

print(
    "매칭률:",
    round(
        crosswalk_v2["SCHUL_CODE"]
        .notna()
        .mean()
        * 100,
        2,
    ),
    "%",
)


EDSS 대상: 2621
exact match: 2597
매칭 실패: 24
매칭률: 99.08 %


In [43]:
# ============================================================
# EDSS ↔ SchoolInfo 확정 학교코드 Master 생성
# - 학생수·학급수 exact match에 성공한 학교만 우선 사용한다.
# - 동일 개방ID가 2024·2025에서 같은 SCHUL_CODE로 연결되는지 검증한다.
# - 연도 중 한 번이라도 확실하게 매칭된 관계를 학교 단위 Master로 만든다.
# ============================================================

crosswalk_matched = (
    crosswalk_v2[
        crosswalk_v2["SCHUL_CODE"].notna()
    ][
        [
            "기준연도",
            "개방ID",
            "SCHUL_CODE",
            "학교명",
        ]
    ]
    .copy()
)


code_consistency = (
    crosswalk_matched
    .groupby("개방ID")["SCHUL_CODE"]
    .nunique()
)


print(
    "여러 SCHUL_CODE로 연결된 개방ID:",
    (code_consistency > 1).sum(),
)


crosswalk_master = (
    crosswalk_matched[
        crosswalk_matched["개방ID"].isin(
            code_consistency[
                code_consistency == 1
            ].index
        )
    ][
        [
            "개방ID",
            "SCHUL_CODE",
            "학교명",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "확정 Master 학교 수:",
    len(crosswalk_master),
)

print(
    "개방ID 중복:",
    crosswalk_master["개방ID"]
    .duplicated()
    .sum(),
)

여러 SCHUL_CODE로 연결된 개방ID:

 0
확정 Master 학교 수: 1305
개방ID 중복: 0


In [44]:
# ============================================================
# exact match 실패 학교의 다른 연도 매칭 여부 확인
# - 2024 또는 2025 한 해에서 실패한 개방ID가 다른 연도에서는
#   이미 확정 Master에 존재하는지 확인한다.
# - 동일 개방ID가 확인되면 수치 차이가 있어도 학교코드는 안전하게 복원할 수 있다.
# ============================================================

crosswalk_failed = (
    crosswalk_v2[
        crosswalk_v2["SCHUL_CODE"].isna()
    ][
        [
            "기준연도",
            "개방ID",
            "자치구",
            "학교급구분",
            "학생수",
            "학급수",
        ]
    ]
    .copy()
)


failed_with_master = (
    crosswalk_failed
    .merge(
        crosswalk_master,
        on="개방ID",
        how="left",
    )
)


print(
    "최초 실패:",
    len(failed_with_master),
)

print(
    "다른 연도로 복원 가능:",
    failed_with_master["SCHUL_CODE"]
    .notna()
    .sum(),
)

print(
    "여전히 미해결:",
    failed_with_master["SCHUL_CODE"]
    .isna()
    .sum(),
)


display(
    failed_with_master[
        failed_with_master["SCHUL_CODE"].isna()
    ]
)

최초 실패: 24
다른 연도로 복원 가능: 10
여전히 미해결: 14


,기준연도,개방ID,자치구,학교급구분,학생수,학급수,SCHUL_CODE,학교명
2,2024,3883373618,성북구,고등학교,741,24,NaN,NaN
4,2024,6113909363,강남구,고등학교,463,28,NaN,NaN
5,2024,6327209889,강남구,고등학교,1401,42,NaN,NaN
6,2024,6858356475,종로구,고등학교,919,30,NaN,NaN
7,2024,8151573896,마포구,중학교,883,25,NaN,NaN
9,2024,8396392524,동작구,고등학교,1037,33,NaN,NaN
10,2024,9009569108,동작구,고등학교,753,25,NaN,NaN
13,2025,3883373618,성북구,고등학교,718,22,NaN,NaN
16,2025,6113909363,강남구,고등학교,402,27,NaN,NaN
17,2025,6327209889,강남구,고등학교,1364,42,NaN,NaN


In [45]:
# ============================================================
# EDSS ↔ SchoolInfo 미해결 학교 7개 수동 Crosswalk 생성
# - exact match가 실패한 7개 학교는 학교 특성·부설학교·이전 이력을 통해 식별했다.
# - SCHUL_CODE는 직접 입력하지 않고 school_basic_seoul에서 학교명으로 조회한다.
# - 수동 보정 관계는 이후 재현 가능하도록 명시적인 딕셔너리로 관리한다.
# ============================================================

MANUAL_EDSS_SCHOOL_MAP = {
    3883373618: "경동고등학교",
    6113909363: "청담고등학교",
    6327209889: "경기여자고등학교",
    6858356475: "경복고등학교",
    8151573896: "아현중학교",
    8396392524: "수도여자고등학교",
    9009569108: "영등포고등학교",
}


manual_crosswalk = pd.DataFrame(
    [
        {
            "개방ID": open_id,
            "학교명": school_name,
        }
        for open_id, school_name
        in MANUAL_EDSS_SCHOOL_MAP.items()
    ]
)


manual_crosswalk = (
    manual_crosswalk
    .merge(
        school_basic_seoul[
            [
                "SCHUL_CODE",
                "SCHUL_NM",
            ]
        ],
        left_on="학교명",
        right_on="SCHUL_NM",
        how="left",
        validate="one_to_one",
    )
    .drop(columns="SCHUL_NM")
)


display(manual_crosswalk)

print(
    "SCHUL_CODE 결측:",
    manual_crosswalk["SCHUL_CODE"].isna().sum(),
)

,개방ID,학교명,SCHUL_CODE
0,3883373618,경동고등학교,S010000377
1,6113909363,청담고등학교,S010000542
2,6327209889,경기여자고등학교,S010000376
3,6858356475,경복고등학교,S010000379
4,8151573896,아현중학교,S010001945
5,8396392524,수도여자고등학교,S010000477
6,9009569108,영등포고등학교,S010000497


SCHUL_CODE 결측: 0


In [46]:
# ============================================================
# 수동 Crosswalk을 기존 확정 Master에 추가
# - 기존 1,305개 exact match 관계는 유지한다.
# - 미해결 7개 학교만 추가해 2024~2025 Crosswalk을 완성한다.
# - 개방ID와 SCHUL_CODE가 모두 1:1인지 최종 검증한다.
# ============================================================

crosswalk_master_final = pd.concat(
    [
        crosswalk_master,
        manual_crosswalk[
            [
                "개방ID",
                "SCHUL_CODE",
                "학교명",
            ]
        ],
    ],
    ignore_index=True,
)


print(
    "최종 Master 학교 수:",
    len(crosswalk_master_final),
)

print(
    "개방ID 중복:",
    crosswalk_master_final["개방ID"]
    .duplicated()
    .sum(),
)

print(
    "SCHUL_CODE 중복:",
    crosswalk_master_final["SCHUL_CODE"]
    .duplicated()
    .sum(),
)


assert (
    crosswalk_master_final["개방ID"]
    .duplicated()
    .sum()
    == 0
)

assert (
    crosswalk_master_final["SCHUL_CODE"]
    .duplicated()
    .sum()
    == 0
)

최종 Master 학교 수: 1312
개방ID 중복: 0
SCHUL_CODE 중복: 0


In [47]:
# ============================================================
# 수동 보정 후 2021~2025 학교코드 Coverage 재검증
# - 완성된 Crosswalk을 EDSS 전체 기간에 적용한다.
# - 2024·2025는 100% 매칭을 목표로 한다.
# - 2021~2023에 남는 학교는 해당 시기에만 존재했던 과거 학교로 별도 분석한다.
# ============================================================

edss_with_school_code = (
    edss_school_environment_active
    .drop(
        columns=[
            "SCHUL_CODE",
            "학교명",
        ],
        errors="ignore",
    )
    .merge(
        crosswalk_master_final,
        on="개방ID",
        how="left",
        validate="many_to_one",
    )
)


coverage_final = (
    edss_with_school_code
    .groupby("기준연도")
    .agg(
        전체학교=("개방ID", "size"),
        코드매칭=("SCHUL_CODE", "count"),
    )
)


coverage_final["매칭률"] = (
    coverage_final["코드매칭"]
    / coverage_final["전체학교"]
    * 100
)


display(
    coverage_final.round(2)
)

,전체학교,코드매칭,매칭률
기준연도,,,
2021,1311,1303,99.39
2022,1315,1306,99.32
2023,1314,1307,99.47
2024,1310,1310,100.00
2025,1311,1311,100.00


In [48]:
# ============================================================
# 2021~2023 과거 미매칭 학교 확인
# - 2024·2025 crosswalk은 100% 완성됐으므로 과거에만 남은 학교를 분리한다.
# - 연도별 미매칭 행 수가 아니라 실제 고유 개방ID 수를 확인한다.
# - 폐교·교명변경·개방ID 변경 가능성을 이후 소수 학교만 개별 보정한다.
# ============================================================

historical_unmatched = (
    edss_with_school_code[
        (edss_with_school_code["기준연도"] <= 2023)
        & (edss_with_school_code["SCHUL_CODE"].isna())
    ]
    .copy()
)


print(
    "미매칭 행 수:",
    len(historical_unmatched),
)

print(
    "고유 개방ID 수:",
    historical_unmatched["개방ID"].nunique(),
)


display(
    historical_unmatched[
        [
            "기준연도",
            "개방ID",
            "자치구",
            "학교급구분",
            "설립형태",
            "학생수",
            "학급수",
            "교원수",
        ]
    ]
    .sort_values(
        ["개방ID", "기준연도"]
    )
)

미매칭 행 수: 24
고유 개방ID 수: 9


,기준연도,개방ID,자치구,학교급구분,설립형태,학생수,학급수,교원수
357,2021,3468505620,양천구,중학교,공립,620,27,55
1669,2022,3468505620,양천구,중학교,공립,418,18,45
2984,2023,3468505620,양천구,중학교,공립,228,10,30
1775,2022,4213441967,송파구,고등학교,공립,297,12,31
3090,2023,4213441967,송파구,고등학교,공립,612,23,55
491,2021,4430612385,광진구,초등학교,공립,117,8,16
1804,2022,4430612385,광진구,초등학교,공립,84,8,16
605,2021,5227165699,서초구,중학교,공립,692,27,60
1919,2022,5227165699,서초구,중학교,공립,434,18,46
3233,2023,5227165699,서초구,중학교,공립,216,9,29


## 5. 과거 학교 수동 보정 및 최종 Crosswalk


In [49]:
# ============================================================
# 2021~2023 과거 학교 Crosswalk 수동 보정
# - 2024~2025에 존재하지 않아 자동 Crosswalk을 만들 수 없었던 9개 학교를 연결한다.
# - 학교알리미 기본정보의 SCHUL_CODE와 좌표를 기준으로 명시적으로 관리한다.
# - 덕수고는 현재 학교가 아닌 과거 '덕수고등학교(일반고)' 코드와 연결한다.
# ============================================================

historical_crosswalk_patch = pd.DataFrame(
    [
        [3468505620, "S010000993", "신화중학교"],
        [4213441967, "S010006558", "덕수고등학교(일반고)"],
        [4430612385, "S010002195", "서울화양초등학교"],
        [5227165699, "S010000711", "반포중학교"],
        [6183646798, "S010000754", "서울반포초등학교"],
        [6290236711, "S010000592", "성수공업고등학교"],
        [7734321951, "S010005206", "도봉고등학교"],
        [8941025793, "S010002337", "금호여자중학교"],
        [9611477196, "S010001407", "서울면중초등학교"],
    ],
    columns=[
        "개방ID",
        "SCHUL_CODE",
        "학교명",
    ],
)


display(historical_crosswalk_patch)

,개방ID,SCHUL_CODE,학교명
0,3468505620,S010000993,신화중학교
1,4213441967,S010006558,덕수고등학교(일반고)
2,4430612385,S010002195,서울화양초등학교
3,5227165699,S010000711,반포중학교
4,6183646798,S010000754,서울반포초등학교
5,6290236711,S010000592,성수공업고등학교
6,7734321951,S010005206,도봉고등학교
7,8941025793,S010002337,금호여자중학교
8,9611477196,S010001407,서울면중초등학교


In [50]:
# ============================================================
# EDSS ↔ SchoolInfo 최종 학교 Crosswalk Master 생성
# - 2024~2025에서 검증한 Crosswalk에 과거 9개 학교를 추가한다.
# - 개방ID와 SCHUL_CODE 중복 여부를 확인해 잘못된 학교 연결을 방지한다.
# - 이 Master를 2021~2025 EDSS 전체 기간에 적용한다.
# ============================================================

school_crosswalk_final = pd.concat(
    [
        crosswalk_master_final,
        historical_crosswalk_patch,
    ],
    ignore_index=True,
)


print(
    "최종 Master 학교 수:",
    len(school_crosswalk_final),
)

print(
    "개방ID 중복:",
    school_crosswalk_final["개방ID"]
    .duplicated()
    .sum(),
)

print(
    "SCHUL_CODE 중복:",
    school_crosswalk_final["SCHUL_CODE"]
    .duplicated()
    .sum(),
)


assert (
    school_crosswalk_final["개방ID"]
    .duplicated()
    .sum()
    == 0
)

assert (
    school_crosswalk_final["SCHUL_CODE"]
    .duplicated()
    .sum()
    == 0
)

최종 Master 학교 수: 1321
개방ID 중복: 0
SCHUL_CODE 중복: 0


In [51]:
# ============================================================
# EDSS 2021~2025 학교코드 최종 Coverage 검증
# - 과거 9개 학교까지 보정한 최종 Crosswalk을 전체 EDSS에 적용한다.
# - 모든 연도에서 학교코드 매칭률 100%를 목표로 한다.
# - 여기서 100%가 확인되면 학교 식별 Crosswalk 작업을 종료한다.
# ============================================================

edss_school_environment_final = (
    edss_school_environment_active
    .drop(
        columns=[
            "SCHUL_CODE",
            "학교명",
        ],
        errors="ignore",
    )
    .merge(
        school_crosswalk_final,
        on="개방ID",
        how="left",
        validate="many_to_one",
    )
)


coverage_final = (
    edss_school_environment_final
    .groupby("기준연도")
    .agg(
        전체학교=("개방ID", "size"),
        코드매칭=("SCHUL_CODE", "count"),
    )
)


coverage_final["매칭률"] = (
    coverage_final["코드매칭"]
    / coverage_final["전체학교"]
    * 100
)


display(
    coverage_final.round(2)
)


assert (
    coverage_final["매칭률"]
    == 100
).all()


print(
    "\n✅ 2021~2025 EDSS 학교코드 매칭 완료"
)

,전체학교,코드매칭,매칭률
기준연도,,,
2021,1311,1311,100.0
2022,1315,1315,100.0
2023,1314,1314,100.0
2024,1310,1310,100.0
2025,1311,1311,100.0



✅ 2021~2025 EDSS 학교코드 매칭 완료


In [52]:
# ============================================================
# 2021~2023 과거 학교 위치정보 연결
# - 최종 Crosswalk의 SCHUL_CODE를 이용해 학교알리미 기본정보 좌표를 가져온다.
# - 과거 학교명은 EDSS 기준 이름을 보존하고 현재 API 학교명은 별도로 확인한다.
# - 현재 CLOSE_YN은 과거 연도 활성 여부 판단에 사용하지 않는다.
# ============================================================

historical_school_location_final = (
    historical_crosswalk_patch
    .merge(
        school_basic_seoul[
            [
                "SCHUL_CODE",
                "SCHUL_NM",
                "학교급구분",
                "자치구",
                "LTTUD",
                "LGTUD",
            ]
        ],
        on="SCHUL_CODE",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "SCHUL_NM": "현재학교명",
            "LTTUD": "위도",
            "LGTUD": "경도",
        }
    )
)


historical_school_location_final[
    ["위도", "경도"]
] = (
    historical_school_location_final[
        ["위도", "경도"]
    ]
    .apply(pd.to_numeric, errors="coerce")
)


display(historical_school_location_final)


print(
    "\n좌표 결측:",
    historical_school_location_final[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
)

,개방ID,SCHUL_CODE,학교명,현재학교명,학교급구분,자치구,위도,경도
0,3468505620,S010000993,신화중학교,신화중학교,중학교,양천구,37.541096,126.830105
1,4213441967,S010006558,덕수고등학교(일반고),덕수고등학교(일반고),고등학교,송파구,37.485235,127.146776
2,4430612385,S010002195,서울화양초등학교,서울화양초등학교,초등학교,광진구,37.544397,127.071080
3,5227165699,S010000711,반포중학교,반포중학교,중학교,서초구,37.503372,126.992201
4,6183646798,S010000754,서울반포초등학교,서울반포초등학교,초등학교,서초구,37.503163,126.991025
5,6290236711,S010000592,성수공업고등학교,성수공업고등학교,고등학교,성동구,37.540792,127.051416
6,7734321951,S010005206,도봉고등학교,도봉고등학교,고등학교,도봉구,37.686720,127.042515
7,8941025793,S010002337,금호여자중학교,금호중학교,중학교,중구,37.558025,127.020490
8,9611477196,S010001407,서울면중초등학교,서울면중초등학교,초등학교,중랑구,37.582408,127.096617



좌표 결측: 위도    0
경도    0
dtype: int64


In [53]:
# ============================================================
# EDSS ↔ SchoolInfo 최종 학교 Crosswalk 저장
# - 2024~2025 exact match와 수동 보정을 통해 완성한 학교 식별 Master를 저장한다.
# - 개방ID를 과거 EDSS Key로, SCHUL_CODE를 학교알리미·좌표 연결 Key로 사용한다.
# - 이후 학교환경 Feature 생성 시 반복해서 매칭 작업을 하지 않도록 재사용한다.
# ============================================================

SCHOOL_CROSSWALK_PATH = (
    PROCESSED_DIR
    / "school"
    / "school_crosswalk_2021_2025.parquet"
)

SCHOOL_CROSSWALK_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


school_crosswalk_final.to_parquet(
    SCHOOL_CROSSWALK_PATH,
    index=False,
)


print("저장 완료:", SCHOOL_CROSSWALK_PATH)
print("학교 수:", len(school_crosswalk_final))

저장 완료: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed\school\school_crosswalk_2021_2025.parquet
학교 수: 1321


## 5-1. EDSS 학교환경 보정 및 위치 연결

In [54]:
# ============================================================
# 방송통신중·고 부설 학교의 EDSS 학생수 과다 집계 확인
# - 일부 학교는 EDSS 학생수에 부설 방송통신중·고 학생이 합산되어 있다.
# - 학급수는 SchoolInfo와 동일하므로 학급당학생수가 과대 계산된다.
# - 주간/야간 학생수 컬럼으로도 분리되지 않아 SchoolInfo와 비교해 대상을 찾는다.
# - 2024·2025 겹치는 연도의 EDSS - SchoolInfo 학생수 차이를 사용한다.
# ============================================================

STUDENT_GAP_THRESHOLD = 50


student_gap = (
    edss_school_environment_final[
        edss_school_environment_final["기준연도"]
        .isin([2024, 2025])
    ][
        [
            "기준연도",
            "개방ID",
            "SCHUL_CODE",
            "학생수",
            "학급수",
        ]
    ]
    .merge(
        school_environment_clean[
            [
                "기준연도",
                "SCHUL_CODE",
                "학교명",
                "학생수",
                "학급수",
            ]
        ],
        on=[
            "기준연도",
            "SCHUL_CODE",
        ],
        how="inner",
        suffixes=("_EDSS", "_SchoolInfo"),
        validate="one_to_one",
    )
)


student_gap["학생수차이"] = (
    student_gap["학생수_EDSS"]
    - student_gap["학생수_SchoolInfo"]
)


broadcast_gap = (
    student_gap[
        student_gap["학생수차이"]
        > STUDENT_GAP_THRESHOLD
    ]
    .copy()
)


print(
    "학급수 불일치 행:",
    (
        student_gap["학급수_EDSS"]
        != student_gap["학급수_SchoolInfo"]
    ).sum(),
)

print(
    "학생수 과다 집계 학교:",
    broadcast_gap["개방ID"].nunique(),
)

display(
    broadcast_gap.sort_values(
        ["개방ID", "기준연도"]
    )
)


assert broadcast_gap["개방ID"].nunique() == 6

학급수 불일치 행: 0
학생수 과다 집계 학교: 6


,기준연도,개방ID,SCHUL_CODE,학생수_EDSS,학급수_EDSS,학교명,학생수_SchoolInfo,학급수_SchoolInfo,학생수차이
417,2024,3883373618,S010000377,741,24,경동고등학교,520,24,221
1727,2025,3883373618,S010000377,718,22,경동고등학교,474,22,244
760,2024,6327209889,S010000376,1401,42,경기여자고등학교,1153,42,248
2071,2025,6327209889,S010000376,1364,42,경기여자고등학교,1159,42,205
854,2024,6858356475,S010000379,919,30,경복고등학교,569,30,350
2166,2025,6858356475,S010000379,886,28,경복고등학교,562,28,324
1034,2024,8151573896,S010001945,883,25,아현중학교,609,25,274
2346,2025,8151573896,S010001945,925,26,아현중학교,666,26,259
1067,2024,8396392524,S010000477,1037,33,수도여자고등학교,820,33,217
2379,2025,8396392524,S010000477,971,32,수도여자고등학교,790,32,181


In [55]:
# ============================================================
# 방송통신 부설 학교 학생수 보정
# - 2024·2025: SchoolInfo 학생수(본교 기준)로 대체한다.
# - 2021~2023: SchoolInfo가 없으므로 2024·2025 평균 차이를 빼서 추정한다.
# - 원본 값은 학생수_원본으로 보존하고 학생수_보정여부 Flag를 남긴다.
# - 보정 후 학급당학생수·교원1인당학생수를 다시 계산한다.
# ============================================================

broadcast_offset = (
    broadcast_gap
    .groupby("개방ID")["학생수차이"]
    .mean()
)


edss_school_environment_final["학생수_원본"] = (
    edss_school_environment_final["학생수"]
)

edss_school_environment_final["학생수_보정여부"] = (
    edss_school_environment_final["개방ID"]
    .isin(broadcast_offset.index)
)


estimated_students = (
    edss_school_environment_final["학생수_원본"]
    - edss_school_environment_final["개방ID"]
    .map(broadcast_offset)
    .fillna(0)
)


schoolinfo_students = (
    edss_school_environment_final[
        ["기준연도", "개방ID"]
    ]
    .merge(
        broadcast_gap[
            [
                "기준연도",
                "개방ID",
                "학생수_SchoolInfo",
            ]
        ],
        on=["기준연도", "개방ID"],
        how="left",
        validate="one_to_one",
    )["학생수_SchoolInfo"]
)


edss_school_environment_final["학생수"] = (
    schoolinfo_students
    .fillna(
        pd.Series(
            estimated_students.to_numpy(),
            index=schoolinfo_students.index,
        )
    )
    .round()
    .astype(int)
    .to_numpy()
)


edss_school_environment_final["학급당학생수"] = (
    edss_school_environment_final["학생수"]
    / edss_school_environment_final["학급수"]
)

edss_school_environment_final["교원1인당학생수"] = (
    edss_school_environment_final["학생수"]
    / edss_school_environment_final["교원수"]
)


display(
    edss_school_environment_final[
        edss_school_environment_final["학생수_보정여부"]
    ][
        [
            "기준연도",
            "개방ID",
            "학교명",
            "학생수_원본",
            "학생수",
            "학급수",
            "학급당학생수",
        ]
    ]
    .sort_values(["개방ID", "기준연도"])
)


# 보정 후 2024·2025는 SchoolInfo와 사실상 일치해야 한다.
recheck_gap = (
    edss_school_environment_final[
        edss_school_environment_final["기준연도"]
        .isin([2024, 2025])
    ][["기준연도", "SCHUL_CODE", "학생수"]]
    .merge(
        school_environment_clean[
            ["기준연도", "SCHUL_CODE", "학생수"]
        ],
        on=["기준연도", "SCHUL_CODE"],
        suffixes=("_EDSS", "_SchoolInfo"),
        validate="one_to_one",
    )
)


assert (
    (
        recheck_gap["학생수_EDSS"]
        - recheck_gap["학생수_SchoolInfo"]
    )
    .abs()
    .max()
    <= 1
)

assert (edss_school_environment_final["학생수"] > 0).all()


print("\n✅ 방송통신 부설 학교 학생수 보정 완료")

,기준연도,개방ID,학교명,학생수_원본,학생수,학급수,학급당학생수
417,2021,3883373618,경동고등학교,843,610,26,23.461538
1729,2022,3883373618,경동고등학교,800,568,26,21.846154
3044,2023,3883373618,경동고등학교,766,534,25,21.360000
4357,2024,3883373618,경동고등학교,741,520,24,21.666667
5667,2025,3883373618,경동고등학교,718,474,22,21.545455
761,2021,6327209889,경기여자고등학교,1460,1234,42,29.380952
2076,2022,6327209889,경기여자고등학교,1395,1168,42,27.809524
3389,2023,6327209889,경기여자고등학교,1377,1150,42,27.380952
4700,2024,6327209889,경기여자고등학교,1401,1153,42,27.452381
6011,2025,6327209889,경기여자고등학교,1364,1159,42,27.595238



✅ 방송통신 부설 학교 학생수 보정 완료


In [56]:
# ============================================================
# EDSS 학교환경 연도별 위치 연결
# - 현재 학교 위치(school_location_v2)와 과거 학교 위치를 SCHUL_CODE로 통합한다.
# - 덕수고는 2024년 송파 위례 캠퍼스로 이전하면서 EDSS 개방ID가 캠퍼스 간에 바뀌었다.
#   · 9478733062: 2021~2023 성동구 특성화고 → 2024~ 송파구 일반고
#   · 4213441967: 2022~2023 송파구 일반고(위례 신설)
# - 9478733062의 2021~2023 행은 성동구 옛 캠퍼스이므로 연도별 좌표 Override를 적용한다.
#   옛 캠퍼스 주소: 서울 성동구 왕십리로 199 (Kakao 주소검색 좌표)
# ============================================================

school_location_all = (
    pd.concat(
        [
            school_location_v2[
                ["SCHUL_CODE", "자치구", "위도", "경도"]
            ],
            historical_school_location_final[
                ["SCHUL_CODE", "자치구", "위도", "경도"]
            ],
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="SCHUL_CODE",
        keep="first",
    )
    .rename(
        columns={
            "자치구": "위치_자치구",
        }
    )
)


SCHOOL_LOCATION_OVERRIDES = [
    {
        "개방ID": 9478733062,
        "시작연도": 2021,
        "종료연도": 2023,
        "위치_자치구": "성동구",
        "위도": 37.5552642216916,
        "경도": 127.041141809583,
        "사유": "덕수고 성동 옛 캠퍼스(특성화고)",
    },
]


edss_school_environment_final = (
    edss_school_environment_final
    .drop(
        columns=["위치_자치구", "위도", "경도"],
        errors="ignore",
    )
    .merge(
        school_location_all,
        on="SCHUL_CODE",
        how="left",
        validate="many_to_one",
    )
)

edss_school_environment_final["위치보정여부"] = False


for override in SCHOOL_LOCATION_OVERRIDES:

    mask = (
        (edss_school_environment_final["개방ID"] == override["개방ID"])
        & edss_school_environment_final["기준연도"].between(
            override["시작연도"],
            override["종료연도"],
        )
    )

    edss_school_environment_final.loc[
        mask,
        ["위치_자치구", "위도", "경도"],
    ] = [
        override["위치_자치구"],
        override["위도"],
        override["경도"],
    ]

    edss_school_environment_final.loc[
        mask,
        "위치보정여부",
    ] = True

    print(f"{override['사유']}: {mask.sum()}행 좌표 보정")


print(
    "\n좌표 결측:",
    edss_school_environment_final[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
    .sum(),
)


gu_mismatch = edss_school_environment_final[
    edss_school_environment_final["자치구"]
    != edss_school_environment_final["위치_자치구"]
]

print("EDSS 자치구 ≠ 위치 자치구:")
display(
    gu_mismatch
    .groupby(
        ["개방ID", "학교명", "자치구", "위치_자치구"]
    )["기준연도"]
    .apply(list)
    .to_frame()
)


assert (
    edss_school_environment_final[
        ["위도", "경도"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

# 청담고는 학교알리미가 서초구로 분류하지만 좌표(강남구 압구정로 419)는
# EDSS 자치구(강남구)와 일치하므로 정상이다. 그 외 불일치는 없어야 한다.
assert set(gu_mismatch["개방ID"]) == {6113909363}


print("\n✅ EDSS 학교환경 위치 연결 완료")

덕수고 성동 옛 캠퍼스(특성화고): 3행 좌표 보정



좌표 결측: 0
EDSS 자치구 ≠ 위치 자치구:


,,,,기준연도
개방ID,학교명,자치구,위치_자치구,
6113909363,청담고등학교,강남구,서초구,"[2021, 2022, 2023, 2024, 2025]"



✅ EDSS 학교환경 위치 연결 완료


In [57]:
# ============================================================
# EDSS 학교환경 2021~2025 최종 테이블 저장
# - Crosswalk·학생수 보정·연도별 위치 보정을 모두 반영한 학교 단위 연도별 테이블이다.
# - 이후 단지 반경 기반 학교환경 Feature는 이 파일에서 계산한다.
# - 보정 내역은 학생수_원본 / 학생수_보정여부 / 위치보정여부 컬럼으로 추적한다.
# ============================================================

EDSS_SCHOOL_ENV_COLS = [
    "기준연도",
    "개방ID",
    "SCHUL_CODE",
    "학교명",
    "자치구",
    "학교급구분",
    "설립형태",
    "학생수",
    "학생수_원본",
    "학생수_보정여부",
    "학급수",
    "교원수",
    "학급당학생수",
    "교원1인당학생수",
    "위도",
    "경도",
    "위치보정여부",
]


edss_school_environment_save = (
    edss_school_environment_final[
        EDSS_SCHOOL_ENV_COLS
    ]
    .sort_values(["기준연도", "개방ID"])
    .reset_index(drop=True)
)


assert (
    edss_school_environment_save
    .duplicated(subset=["기준연도", "개방ID"])
    .sum()
    == 0
)

assert (
    edss_school_environment_save
    .duplicated(subset=["기준연도", "SCHUL_CODE"])
    .sum()
    == 0
)


EDSS_SCHOOL_ENV_PATH = (
    PROCESSED_DIR
    / "school"
    / "edss_school_environment_2021_2025.parquet"
)

edss_school_environment_save.to_parquet(
    EDSS_SCHOOL_ENV_PATH,
    index=False,
)


print("저장 완료:", EDSS_SCHOOL_ENV_PATH)
print("shape:", edss_school_environment_save.shape)

display(
    edss_school_environment_save
    .groupby("기준연도")
    .agg(
        학교수=("개방ID", "size"),
        학생수보정=("학생수_보정여부", "sum"),
        위치보정=("위치보정여부", "sum"),
    )
)

저장 완료: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed\school\edss_school_environment_2021_2025.parquet
shape: (6561, 17)


,학교수,학생수보정,위치보정
기준연도,,,
2021,1311,6,1
2022,1315,6,1
2023,1314,6,1
2024,1310,6,0
2025,1311,6,0


## 6. 추가 교육환경 Feature 후보 탐색


In [58]:
# ============================================================
# EDSS 학교개황 추가 교육환경 후보 컬럼 탐색
# - 학생수·학급수·교원수 외에 학교 시설·공간·입학·졸업 관련 지표를 확인한다.
# - 컬럼을 바로 Feature로 사용하지 않고 부동산 교육입지와 관련 있는 후보만 선별한다.
# - 중복되거나 의미가 불명확한 컬럼은 이후 EDA에서 제외한다.
# ============================================================

candidate_keywords = [
    "학생",
    "교원",
    "입학",
    "졸업",
    "교실",
    "공간",
    "면적",
    "체육",
    "용지",
    "시설",
]


school_overview_candidate_cols = [
    col
    for col in edss_school_overview.columns
    if any(
        keyword in col
        for keyword in candidate_keywords
    )
]


print(
    "후보 컬럼 수:",
    len(school_overview_candidate_cols),
)

for col in school_overview_candidate_cols:
    print(col)

후보 컬럼 수: 37
유초중등학교개황_전공과정여학생수
유초중등학교개황_전공과정학생수
고등학교과정여학생수
고등학교과정학생수
중학교과정여학생수
중학교과정학생수
초등학교과정여학생수
초등학교과정학생수
유초중등학교개황_유치원과정여학생수
유초중등학교개황_유치원과정학생수
유초중등학교개황_3학년야간여학생수
유초중등학교개황_3학년야간학생수
유초중등학교개황_2학년야간여학생수
유초중등학교개황_2학년야간학생수
유초중등학교개황_1학년야간여학생수
유초중등학교개황_1학년야간학생수
유초중등학교개황_3학년주간여학생수
유초중등학교개황_3학년주간학생수
유초중등학교개황_기타교사용도면적
유초중등학교개황_관리지원공간면적
유초중등학교개황_교수학습공간면적
유초중등학교개황_학교용지체육장면적
유초중등학교개황_학교용지대지면적
가교실및대용교실수
유초중등학교개황_정규교실수
유초중등학교개황_여자교원수
유초중등학교개황_교원수
유초중등학교개황_여자졸업생수
유초중등학교개황_졸업생수
유초중등학교개황_여자입학생수
유초중등학교개황_입학생수
유초중등학교개황_2학년주간여학생수
유초중등학교개황_2학년주간학생수
유초중등학교개황_1학년주간여학생수
유초중등학교개황_1학년주간학생수
유초중등학교개황_여학생수
유초중등학교개황_학생수


학교 규모/수요
- 학생수
- 학생수 전년증감률
- 입학생수
- 졸업생수

과밀/교원
- 학급당학생수
- 교원1인당학생수
- 교원수
- 교원 구성

시설
- 학생1인당 교수학습공간
- 학생1인당 체육장면적
- 학생1인당 학교대지면적
- 학급당 정규교실수

학습지원
- 도서관 장서수
- 학생1인당 장서수
- 도서관 좌석수
- 방과후 프로그램
- 돌봄 관련 지표

학교 공급/접근성
- 500m 내 초등학교수
- 1km 내 중학교수
- 1km 내 고등학교수
- 최근접 학교 거리

사교육
- 500m/1km 내 학원수
- 입시·보습학원수
- 학원 밀도